In [11]:
import pandas as pd
dataset=pd.read_csv("PrePlacement.csv")
dataset.head()

# Covariance

In [12]:
dataset.isnull().sum()
dataset.cov(numeric_only=True)

ValueError: could not convert string to float: 'M'

# Correlation

In [10]:
dataset.corr(numeric_only=True)

,sl_no,ssc_p,hsc_p,degree_p,etest_p,mba_p,salary
sl_no,1.000000,-0.078155,-0.085711,-0.088281,0.063636,0.022327,0.063764
ssc_p,-0.078155,1.000000,0.511472,0.538404,0.261993,0.388478,0.035330
hsc_p,-0.085711,0.511472,1.000000,0.434206,0.245113,0.354823,0.076819
degree_p,-0.088281,0.538404,0.434206,1.000000,0.224470,0.402364,-0.019272
etest_p,0.063636,0.261993,0.245113,0.224470,1.000000,0.218055,0.178307
mba_p,0.022327,0.388478,0.354823,0.402364,0.218055,1.000000,0.175013
salary,0.063764,0.035330,0.076819,-0.019272,0.178307,0.175013,1.000000


In [ ]:
dataset.corrwith(dataset["mba_p"])

# Preprocessing

In [ ]:
dataset=dataset.drop(columns='sl_no')
dataset=dataset.dropna()

# Visulaize the pair

In [ ]:
import seaborn as sns
sns.pairplot(dataset,hue="ssc_b")

import matplotlib.pyplot as plt
plt.savefig('pairplot_ssc_b.png')

# Calling user defined function Descriptive file

In [ ]:
from Descriptive import Descriptive

obj=Descriptive()

# Sregrating quantitative Data and qualtative

In [ ]:
quan,qual=obj.segreQuanQual(dataset)

In [ ]:
quan_data=dataset[quan]

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calc_vif(X):

    # Calculating VIF
    vif = pd.DataFrame()
    vif["variables"] = X.columns
    vif["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

    return(vif)

In [ ]:
calc_vif(quan_data)

In [ ]:
quan_data=dataset[['etest_p','salary',]]

# Univarite Analysis

## Probability Density Function(PDF)

In [ ]:
# Density Polt

import seaborn as sns
x=dataset['ssc_p']
ax = sns.distplot(dataset['ssc_p'],kde=True)

In [ ]:
def get_pdf_probability(dataset,startrange,endrange):
    from matplotlib import pyplot
    from scipy.stats import norm
    import seaborn as sns
    ax = sns.distplot(dataset,kde=True,kde_kws={'color':'blue'},color='Green')
    pyplot.axvline(startrange,color='Red')
    pyplot.axvline(endrange,color='Red')
    # generate a sample
    sample = dataset
    # calculate parameters
    sample_mean =sample.mean()
    sample_std = sample.std()
    print('Mean=%.3f, Standard Deviation=%.3f' % (sample_mean, sample_std))
    # define the distribution
    dist = norm(sample_mean, sample_std)
    
    # sample probabilities for a range of outcomes
    values = [value for value in range(startrange, endrange)]
    probabilities = [dist.pdf(value) for value in values]    
    prob=sum(probabilities)
    print("The area between range({},{}):{}".format(startrange,endrange,sum(probabilities)))
    return prob
    

In [ ]:
get_pdf_probability(dataset['ssc_p'],60,60)

# cumulative Density function

In [ ]:
import matplotlib.pyplot as plt
x=dataset['ssc_p']
plt.hist(x,cumulative=True, density=True,bins=100)

In [ ]:
from statsmodels.distributions.empirical_distribution import ECDF
ecdf = ECDF(dataset['ssc_p'])
ecdf(65)

# Types of Test

# Z- Score: Normal distribution to Standard Normal Distribution

In [ ]:
# Normal Distribution
import seaborn as sns
sns.distplot(dataset['ssc_p'],kde=True)

In [ ]:
# Coverted to standard Normal Distribution
import seaborn as sns
mean=dataset['ssc_p'].mean()
std=dataset['ssc_p'].std()

values=[i for i in dataset['ssc_p']]

z_score=[((j-mean)/std) for j in values]

sns.distplot(z_score,kde=True)

sum(z_score)/len(z_score)
#z_score.std()


In [ ]:
dataset

In [ ]:
dataset[dataset["gender"]=="M"]["salary"]

# T-Test

#### Independant Sample
Diferrent Group(Male, Female) but same contion(salary)

In [ ]:
from scipy.stats import ttest_ind
dataset=dataset.dropna()
male = dataset[dataset['gender']=='M']['salary']
female = dataset[dataset['gender']=='F']['salary']
#print(male)
ttest_ind(male, female)

#### Dependant Sample
Same Group(Male) but Different Condition(ssc_p,hsc_p)


In [ ]:
from scipy.stats import ttest_rel
#dataset=dataset.dropna()
male = dataset[dataset['gender']=='M']['ssc_p']
male1 = dataset[dataset['gender']=='M']['hsc_p']
ttest_rel(male, male1)


### for one sample mean

should same for different sample

scipy.stats.ttest_1samp(a, popmean, axis=0, nan_policy='propagate')

# ANAVO : Analysis of Variance

## One-Way Classification

In [ ]:
import scipy.stats as stats

stats.f_oneway(dataset['ssc_p'],dataset['hsc_p'],dataset['degree_p'])


In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

#perform two-way ANOVA
model = ols('degree_p ~ C(ssc_p) + C(hsc_p) + C(ssc_p):C(hsc_p)', data=dataset).fit()
sm.stats.anova_lm(model, typ=2)